In [1]:
# LangChain RAG Walkthrough with ChromaDB

This notebook demonstrates a complete Retrieval-Augmented Generation (RAG) implementation using:
- **LangChain** for orchestration
- **ChromaDB** as the vector store (ephemeral, no background processes)
- **Flexible LLM support** for OpenAI, Gemini, or Ollama
- **Document chunking** and **embedding** workflows
- **Similarity-based retrieval**

## Overview
1. Setup and Dependencies
2. Document Loading and Chunking
3. Embedding Generation
4. Vector Store Creation (ChromaDB)
5. Similarity Retrieval
6. LLM Integration (OpenAI/Gemini/Ollama)
7. Complete RAG Pipeline

SyntaxError: invalid syntax (2235185487.py, line 3)

In [2]:
# Install required packages
!pip install langchain langchain-community langchain-openai langchain-google-genai chromadb sentence-transformers pypdf2 python-dotenv bs4

In [3]:
# Import required libraries
import os
from typing import List, Optional
from dotenv import load_dotenv

# LangChain imports
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.docstore.document import Document
from langchain_community.vectorstores import Chroma
from langchain.schema import BaseRetriever
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

# Embedding models
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_openai import OpenAIEmbeddings

# LLM models
from langchain_openai import ChatOpenAI
from langchain_community.llms import Ollama

# Load environment variables
load_dotenv()

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


## 1. Document Preparation and Chunking

We'll start by creating some sample documents and demonstrating how to chunk them effectively for RAG.

In [4]:
# Create sample documents for demonstration
sample_documents = [
    """
    Machine Learning is a subset of artificial intelligence that enables computers to learn and improve 
    from experience without being explicitly programmed. It focuses on the development of computer programs 
    that can access data and use it to learn for themselves. The process of learning begins with observations 
    or data, such as examples, direct experience, or instruction, in order to look for patterns in data and 
    make better decisions in the future based on the examples that we provide.
    """,
    """
    Deep Learning is a subset of machine learning that uses neural networks with multiple layers (hence "deep") 
    to model and understand complex patterns in data. These neural networks attempt to simulate the behavior 
    of the human brain, allowing it to "learn" from large amounts of data. Deep learning has been particularly 
    successful in areas such as image recognition, natural language processing, and speech recognition.
    """,
    """
    Natural Language Processing (NLP) is a branch of artificial intelligence that deals with the interaction 
    between computers and humans through natural language. The ultimate objective of NLP is to read, decipher, 
    understand, and make sense of human languages in a manner that is valuable. NLP combines computational 
    linguistics with statistical, machine learning, and deep learning models to help computers process human language.
    """,
    """
    Retrieval-Augmented Generation (RAG) is an AI framework that combines the strengths of parametric and 
    non-parametric knowledge. It retrieves relevant information from a knowledge base and uses that information 
    to generate more accurate and contextually relevant responses. RAG models first retrieve relevant documents 
    from a corpus using a retriever, then use a generator to produce the final output conditioned on both the 
    query and the retrieved documents.
    """
]

# Convert to LangChain Document objects
documents = [Document(page_content=doc.strip(), metadata={"source": f"doc_{i}"}) 
             for i, doc in enumerate(sample_documents)]

print(f"Created {len(documents)} sample documents")
for i, doc in enumerate(documents):
    print(f"Document {i}: {len(doc.page_content)} characters")
    print(f"Preview: {doc.page_content[:100]}...")
    print("---")

Created 4 sample documents
Document 0: 508 characters
Preview: Machine Learning is a subset of artificial intelligence that enables computers to learn and improve ...
---
Document 1: 434 characters
Preview: Deep Learning is a subset of machine learning that uses neural networks with multiple layers (hence ...
---
Document 2: 444 characters
Preview: Natural Language Processing (NLP) is a branch of artificial intelligence that deals with the interac...
---
Document 3: 478 characters
Preview: Retrieval-Augmented Generation (RAG) is an AI framework that combines the strengths of parametric an...
---


In [5]:
# Demonstrate different chunking strategies
def show_chunking_strategies():
    # Strategy 1: Small chunks
    small_splitter = RecursiveCharacterTextSplitter(
        chunk_size=200,
        chunk_overlap=50,
        length_function=len,
    )
    
    # Strategy 2: Medium chunks
    medium_splitter = RecursiveCharacterTextSplitter(
        chunk_size=400,
        chunk_overlap=100,
        length_function=len,
    )
    
    # Strategy 3: Large chunks
    large_splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=150,
        length_function=len,
    )
    
    # Test with the first document
    test_doc = documents[0]
    
    strategies = [
        ("Small (200 chars, 50 overlap)", small_splitter),
        ("Medium (400 chars, 100 overlap)", medium_splitter),
        ("Large (800 chars, 150 overlap)", large_splitter)
    ]
    
    for name, splitter in strategies:
        chunks = splitter.split_documents([test_doc])
        print(f"\n{name}:")
        print(f"Number of chunks: {len(chunks)}")
        for i, chunk in enumerate(chunks):
            print(f"  Chunk {i+1}: {len(chunk.page_content)} chars - '{chunk.page_content[:50]}...'")

show_chunking_strategies()


Small (200 chars, 50 overlap):
Number of chunks: 4
  Chunk 1: 99 chars - 'Machine Learning is a subset of artificial intelli...'
  Chunk 2: 103 chars - 'from experience without being explicitly programme...'
  Chunk 3: 105 chars - 'that can access data and use it to learn for thems...'
  Chunk 4: 183 chars - 'or data, such as examples, direct experience, or i...'

Medium (400 chars, 100 overlap):
Number of chunks: 2
  Chunk 1: 319 chars - 'Machine Learning is a subset of artificial intelli...'
  Chunk 2: 183 chars - 'or data, such as examples, direct experience, or i...'

Large (800 chars, 150 overlap):
Number of chunks: 1
  Chunk 1: 508 chars - 'Machine Learning is a subset of artificial intelli...'


## 2. Web Content Loading

LangChain provides powerful web loaders to extract content from websites. We'll demonstrate how to use the WebBaseLoader to load HTML content from websites and prepare it for RAG.

In [6]:
# Import web loading libraries
from langchain_community.document_loaders import WebBaseLoader
from bs4 import BeautifulSoup
import requests
from urllib.parse import urljoin, urlparse
import time

print("✅ Web loading libraries imported successfully!")

USER_AGENT environment variable not set, consider setting it to identify your requests.


✅ Web loading libraries imported successfully!


In [7]:
# Create final chunks for our RAG pipeline
# Using medium-sized chunks for optimal balance
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=100,
    length_function=len,
)

# Split all documents
all_chunks = text_splitter.split_documents(documents)

print(f"✅ Created {len(all_chunks)} chunks from {len(documents)} documents")
print("\nChunk details:")
for i, chunk in enumerate(all_chunks):
    source = chunk.metadata.get('source', 'unknown')
    print(f"Chunk {i+1}: {len(chunk.page_content)} chars from {source}")
    print(f"Preview: {chunk.page_content[:80]}...")
    print("---")

✅ Created 8 chunks from 4 documents

Chunk details:
Chunk 1: 319 chars from doc_0
Preview: Machine Learning is a subset of artificial intelligence that enables computers t...
---
Chunk 2: 183 chars from doc_0
Preview: or data, such as examples, direct experience, or instruction, in order to look f...
---
Chunk 3: 329 chars from doc_1
Preview: Deep Learning is a subset of machine learning that uses neural networks with mul...
---
Chunk 4: 99 chars from doc_1
Preview: successful in areas such as image recognition, natural language processing, and ...
---
Chunk 5: 324 chars from doc_2
Preview: Natural Language Processing (NLP) is a branch of artificial intelligence that de...
---
Chunk 6: 114 chars from doc_2
Preview: linguistics with statistical, machine learning, and deep learning models to help...
---
Chunk 7: 327 chars from doc_3
Preview: Retrieval-Augmented Generation (RAG) is an AI framework that combines the streng...
---
Chunk 8: 145 chars from doc_3
Preview: from a corpus using a

## 2. Embedding Strategies

We'll demonstrate different embedding approaches that work with various providers.

In [8]:
# Configure different embedding models
def get_embedding_model(provider: str = "ollama"):
    """
    Get embedding model based on provider choice.
    Options: "openai", "ollama"
    """
    if provider == "openai":
        # Requires OPENAI_API_KEY environment variable
        return OpenAIEmbeddings(model="text-embedding-3-small")
    
    elif provider == "ollama":
        # Free, local embedding model using Ollama (no API key required)
        # Requires: ollama pull nomic-embed-text
        from langchain_community.embeddings import OllamaEmbeddings
        return OllamaEmbeddings(
            model="nomic-embed-text",
            base_url="http://localhost:11434"
        )
    
    else:
        raise ValueError(f"Unsupported provider: {provider}. Use 'openai' or 'ollama'")

# Demonstrate embedding creation
print("Available embedding providers:")
print("1. ollama (free, local - requires: ollama pull nomic-embed-text)")
print("2. openai (requires OPENAI_API_KEY)")

# Use Ollama by default (no API key required, but requires ollama setup)
try:
    embeddings = get_embedding_model("ollama")
    print(f"\n✅ Using Ollama embeddings: {embeddings}")
    
    # Test embedding a sample text
    sample_text = "This is a test sentence for embedding."
    sample_embedding = embeddings.embed_query(sample_text)
    print(f"\nSample embedding dimensions: {len(sample_embedding)}")
    print(f"First 5 values: {sample_embedding[:5]}")
except Exception as e:
    print(f"\n⚠️ Ollama embeddings not available: {e}")
    print("To use Ollama embeddings:")
    print("1. Install Ollama: https://ollama.ai/")
    print("2. Pull embedding model: ollama pull nomic-embed-text")
    print("3. Ensure Ollama is running")
    print("\nFalling back to OpenAI embeddings (requires API key)")
    embeddings = None

Available embedding providers:
1. ollama (free, local - requires: ollama pull nomic-embed-text)
2. openai (requires OPENAI_API_KEY)

✅ Using Ollama embeddings: base_url='http://localhost:11434' model='nomic-embed-text' embed_instruction='passage: ' query_instruction='query: ' mirostat=None mirostat_eta=None mirostat_tau=None num_ctx=None num_gpu=None num_thread=None repeat_last_n=None repeat_penalty=None temperature=None stop=None tfs_z=None top_k=None top_p=None show_progress=False headers=None model_kwargs=None


/tmp/ipykernel_15265/1245328251.py:15: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaEmbeddings``.
  return OllamaEmbeddings(



Sample embedding dimensions: 768
First 5 values: [1.0873792171478271, 1.6060736179351807, -3.63159441947937, -1.6120625734329224, 0.9606084823608398]


## 3. ChromaDB Vector Store

Now we'll create an ephemeral ChromaDB vector store and embed our documents.

In [9]:
# Create ChromaDB vector store (ephemeral - no persistence)
print("Creating ChromaDB vector store...")

# Create the vector store with our chunks and embeddings
vectorstore = Chroma.from_documents(
    documents=all_chunks,
    embedding=embeddings,
    collection_name="rag_demo",
    # No persist_directory = ephemeral (in-memory only)
)

print(f"✅ Vector store created with {vectorstore._collection.count()} documents")

# Verify the embedding process worked
collection_info = vectorstore._collection.count()
print(f"Collection contains {collection_info} embedded documents")

# Show some collection metadata
try:
    # Peek at the collection to verify it's working
    peek_result = vectorstore._collection.peek(limit=2)
    print(f"Sample IDs: {peek_result['ids'][:2] if peek_result['ids'] else 'None'}")
    print(f"Sample metadata keys: {list(peek_result['metadatas'][0].keys()) if peek_result['metadatas'] else 'None'}")
except Exception as e:
    print(f"Note: {e}")

Creating ChromaDB vector store...
✅ Vector store created with 8 documents
Collection contains 8 embedded documents
Sample IDs: ['8412f3c2-bd29-477d-8660-b93442af2b50', '057c86dc-dc59-4cde-b19c-d832bb19eb0d']
Sample metadata keys: ['source']
✅ Vector store created with 8 documents
Collection contains 8 embedded documents
Sample IDs: ['8412f3c2-bd29-477d-8660-b93442af2b50', '057c86dc-dc59-4cde-b19c-d832bb19eb0d']
Sample metadata keys: ['source']


## 4. Similarity Retrieval

Let's test the retrieval functionality with different queries and similarity thresholds.

In [10]:
# Demonstrate similarity search
def test_similarity_search(query: str, k: int = 3):
    """Test similarity search with a given query"""
    print(f"\n🔍 Query: '{query}'")
    print(f"Retrieving top {k} similar documents:")
    print("-" * 50)
    
    # Similarity search
    results = vectorstore.similarity_search(query, k=k)
    
    for i, doc in enumerate(results, 1):
        source = doc.metadata.get('source', 'unknown')
        print(f"{i}. Source: {source}")
        print(f"   Content: {doc.page_content[:150]}...")
        print()
    
    return results

# Test with different queries
test_queries = [
    "What is machine learning?",
    "neural networks and deep learning",
    "How does RAG work?",
    "natural language processing applications"
]

for query in test_queries:
    test_similarity_search(query, k=2)


🔍 Query: 'What is machine learning?'
Retrieving top 2 similar documents:
--------------------------------------------------
1. Source: doc_0
   Content: Machine Learning is a subset of artificial intelligence that enables computers to learn and improve 
    from experience without being explicitly prog...

2. Source: doc_1
   Content: Deep Learning is a subset of machine learning that uses neural networks with multiple layers (hence "deep") 
    to model and understand complex patte...


🔍 Query: 'neural networks and deep learning'
Retrieving top 2 similar documents:
--------------------------------------------------
1. Source: doc_1
   Content: Deep Learning is a subset of machine learning that uses neural networks with multiple layers (hence "deep") 
    to model and understand complex patte...

2. Source: doc_2
   Content: linguistics with statistical, machine learning, and deep learning models to help computers process human language....


🔍 Query: 'How does RAG work?'
Retrieving 

In [11]:
# Demonstrate similarity search with scores
def test_similarity_search_with_scores(query: str, k: int = 3):
    """Test similarity search with similarity scores"""
    print(f"\n📊 Query with scores: '{query}'")
    print("-" * 50)
    
    # Similarity search with scores
    results = vectorstore.similarity_search_with_score(query, k=k)
    
    for i, (doc, score) in enumerate(results, 1):
        source = doc.metadata.get('source', 'unknown')
        print(f"{i}. Similarity Score: {score:.4f}")
        print(f"   Source: {source}")
        print(f"   Content: {doc.page_content[:100]}...")
        print()
    
    return results

# Test with scores
test_similarity_search_with_scores("What is deep learning?", k=3)

# Test retriever functionality
print("\n🔄 Testing retriever interface:")
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
retrieved_docs = retriever.get_relevant_documents("machine learning algorithms")

print(f"Retrieved {len(retrieved_docs)} documents using retriever interface")
for i, doc in enumerate(retrieved_docs, 1):
    print(f"{i}. {doc.page_content[:80]}...")


📊 Query with scores: 'What is deep learning?'
--------------------------------------------------
1. Similarity Score: 261.7079
   Source: doc_1
   Content: Deep Learning is a subset of machine learning that uses neural networks with multiple layers (hence ...

2. Similarity Score: 389.8978
   Source: doc_0
   Content: Machine Learning is a subset of artificial intelligence that enables computers to learn and improve ...

3. Similarity Score: 422.0031
   Source: doc_2
   Content: linguistics with statistical, machine learning, and deep learning models to help computers process h...


🔄 Testing retriever interface:
Retrieved 2 documents using retriever interface
1. Machine Learning is a subset of artificial intelligence that enables computers t...
2. linguistics with statistical, machine learning, and deep learning models to help...


/tmp/ipykernel_15265/588553717.py:25: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  retrieved_docs = retriever.get_relevant_documents("machine learning algorithms")


## 5. LLM Integration (OpenAI/Gemini/Ollama)

Now we'll set up flexible LLM integration that can work with different providers.

In [12]:
# Configure different LLM providers
def get_llm(provider: str = "ollama", model: str = None):
    """
    Get LLM based on provider choice.
    Options: "openai", "ollama"
    """
    if provider == "openai":
        # Requires OPENAI_API_KEY environment variable
        model = model or "gpt-3.5-turbo"
        return ChatOpenAI(
            model=model,
            temperature=0.1,
            max_tokens=1000
        )
    
    elif provider == "ollama":
        # Requires Ollama to be running locally
        # Install: https://ollama.ai/
        # Run: ollama pull llama3.2:1b (or your preferred model)
        model = model or "llama3.2:1b"  # Using the lightweight 1B parameter model
        return Ollama(
            model=model,
            temperature=0.1
        )
    
    else:
        raise ValueError(f"Unsupported provider: {provider}. Use 'openai' or 'ollama'")

# Display available options
print("Available LLM providers:")
print("1. ollama (requires local Ollama installation)")
print("   - Models: llama3.2:1b, llama2, mistral, codellama, etc.")
print("   - Install: https://ollama.ai/")
print("2. openai (requires OPENAI_API_KEY)")
print("   - Models: gpt-3.5-turbo, gpt-4, etc.")

# Example LLM configuration (choose based on your setup)
try:
    # Try Ollama first with the lightweight llama3.2:1b model
    llm = get_llm("ollama", "llama3.2:1b")
    print(f"\n✅ Using Ollama with llama3.2:1b")
    llm_provider = "ollama"
except Exception as e:
    print(f"\n⚠️ Ollama not available: {e}")
    print("You can:")
    print("1. Install Ollama: https://ollama.ai/")
    print("2. Pull the model: ollama pull llama3.2:1b")
    print("3. Set up OpenAI API key in environment")
    llm = None
    llm_provider = None

Available LLM providers:
1. ollama (requires local Ollama installation)
   - Models: llama3.2:1b, llama2, mistral, codellama, etc.
   - Install: https://ollama.ai/
2. openai (requires OPENAI_API_KEY)
   - Models: gpt-3.5-turbo, gpt-4, etc.

✅ Using Ollama with llama3.2:1b


/tmp/ipykernel_15265/2268038758.py:21: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaLLM``.
  return Ollama(


## 6. Complete RAG Pipeline

Now let's put it all together into a complete RAG system!

In [13]:
# Complete RAG Pipeline Class
class RAGPipeline:
    def __init__(self, embedding_provider="ollama", llm_provider="ollama", llm_model=None):
        """
        Initialize RAG pipeline with flexible provider options.
        
        Args:
            embedding_provider: "ollama" or "openai"
            llm_provider: "ollama" or "openai"  
            llm_model: Specific model name (optional)
        """
        self.embedding_provider = embedding_provider
        self.llm_provider = llm_provider
        
        # Initialize embeddings
        self.embeddings = get_embedding_model(embedding_provider)
        
        # Initialize LLM
        try:
            self.llm = get_llm(llm_provider, llm_model)
        except Exception as e:
            print(f"Warning: Could not initialize LLM ({llm_provider}): {e}")
            self.llm = None
        
        self.vectorstore = None
        self.retriever = None
        self.qa_chain = None
        
    def create_vectorstore(self, documents, collection_name="rag_collection"):
        """Create ephemeral ChromaDB vector store from documents."""
        print(f"Creating vector store with {len(documents)} documents...")
        
        self.vectorstore = Chroma.from_documents(
            documents=documents,
            embedding=self.embeddings,
            collection_name=collection_name
        )
        
        # Create retriever
        self.retriever = self.vectorstore.as_retriever(
            search_type="similarity",
            search_kwargs={"k": 3}
        )
        
        print(f"✅ Vector store created with {self.vectorstore._collection.count()} documents")
        return self.vectorstore
    
    def setup_qa_chain(self):
        """Set up the question-answering chain."""
        if not self.llm:
            raise ValueError("LLM not available. Please configure an LLM provider.")
        
        if not self.retriever:
            raise ValueError("Retriever not available. Please create vector store first.")
        
        # Custom prompt template
        prompt_template = """Use the following pieces of context to answer the question at the end. 
        If you don't know the answer, just say that you don't know, don't try to make up an answer.

        Context:
        {context}

        Question: {question}
        
        Answer:"""
        
        PROMPT = PromptTemplate(
            template=prompt_template,
            input_variables=["context", "question"]
        )
        
        # Create QA chain
        self.qa_chain = RetrievalQA.from_chain_type(
            llm=self.llm,
            chain_type="stuff",
            retriever=self.retriever,
            chain_type_kwargs={"prompt": PROMPT},
            return_source_documents=True
        )
        
        print("✅ QA chain configured")
        return self.qa_chain
    
    def query(self, question: str, return_sources: bool = True):
        """Query the RAG system."""
        if not self.qa_chain:
            raise ValueError("QA chain not configured. Please run setup_qa_chain() first.")
        
        print(f"\n🤖 Question: {question}")
        print("-" * 50)
        
        result = self.qa_chain({"query": question})
        
        answer = result["result"]
        sources = result.get("source_documents", [])
        
        print(f"Answer: {answer}")
        
        if return_sources and sources:
            print(f"\n📚 Sources ({len(sources)} documents):")
            for i, doc in enumerate(sources, 1):
                source = doc.metadata.get('source', 'unknown')
                print(f"{i}. {source}: {doc.page_content[:100]}...")
        
        return {
            "question": question,
            "answer": answer,
            "sources": sources
        }

# Initialize the RAG pipeline
rag = RAGPipeline(
    embedding_provider="ollama",  # Local Ollama embeddings
    llm_provider="ollama",  # Change to "openai" if you have API key
    llm_model="llama3.2:1b"  # Using the lightweight 1B parameter model
)

print(f"✅ RAG Pipeline initialized")
print(f"   Embeddings: {rag.embedding_provider}")
print(f"   LLM: {rag.llm_provider} ({'available' if rag.llm else 'not available'})")
print(f"   Model: llama3.2:1b (lightweight, fast responses)")

✅ RAG Pipeline initialized
   Embeddings: ollama
   LLM: ollama (available)
   Model: llama3.2:1b (lightweight, fast responses)


In [14]:
# Set up the complete RAG system
print("Setting up complete RAG system...")

# Create vector store with our chunks
rag.create_vectorstore(all_chunks, "ai_knowledge_base")

# Set up QA chain (only if LLM is available)
if rag.llm:
    rag.setup_qa_chain()
    print("🚀 RAG system ready for queries!")
    print("💡 Using llama3.2:1b - a lightweight, fast model perfect for experimentation")
else:
    print("⚠️ LLM not available - you can still test retrieval functionality")
    print("To enable full RAG:")
    print("1. Install Ollama and pull a model: 'ollama pull llama3.2:1b'")
    print("2. Or set OPENAI_API_KEY for OpenAI")

Setting up complete RAG system...
Creating vector store with 8 documents...
✅ Vector store created with 8 documents
✅ QA chain configured
🚀 RAG system ready for queries!
💡 Using llama3.2:1b - a lightweight, fast model perfect for experimentation
✅ Vector store created with 8 documents
✅ QA chain configured
🚀 RAG system ready for queries!
💡 Using llama3.2:1b - a lightweight, fast model perfect for experimentation


In [15]:
# Test the complete RAG system
if rag.llm and rag.qa_chain:
    print("🧪 Testing complete RAG pipeline...")
    
    # Test queries
    test_questions = [
        "What is machine learning?",
        "How does deep learning differ from traditional machine learning?", 
        "What are the main applications of NLP?",
        "Explain how RAG works"
    ]
    
    for question in test_questions:
        try:
            result = rag.query(question)
            print("\n" + "="*60 + "\n")
        except Exception as e:
            print(f"Error processing question '{question}': {e}")
            break
            
else:
    print("🔍 Testing retrieval functionality only...")
    
    # Test retrieval without LLM
    if rag.retriever:
        test_questions = [
            "What is machine learning?",
            "deep learning neural networks",
            "natural language processing"
        ]
        
        for question in test_questions:
            print(f"\n🔍 Retrieving for: '{question}'")
            docs = rag.retriever.get_relevant_documents(question)
            for i, doc in enumerate(docs, 1):
                source = doc.metadata.get('source', 'unknown')
                print(f"{i}. {source}: {doc.page_content[:100]}...")
            print("-" * 40)

🧪 Testing complete RAG pipeline...

🤖 Question: What is machine learning?
--------------------------------------------------


/tmp/ipykernel_15265/3258127149.py:92: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  result = self.qa_chain({"query": question})


Answer: I don't know.

📚 Sources (3 documents):
1. doc_0: Machine Learning is a subset of artificial intelligence that enables computers to learn and improve ...
2. doc_1: Deep Learning is a subset of machine learning that uses neural networks with multiple layers (hence ...
3. doc_2: linguistics with statistical, machine learning, and deep learning models to help computers process h...



🤖 Question: How does deep learning differ from traditional machine learning?
--------------------------------------------------
Answer: I don't know. Deep learning is a subset of machine learning that uses neural networks with multiple layers (hence "deep") to model and understand complex patterns in data, whereas traditional machine learning typically involves simpler models like statistical or rule-based approaches.

📚 Sources (3 documents):
1. doc_1: Deep Learning is a subset of machine learning that uses neural networks with multiple layers (hence ...
2. doc_0: Machine Learning is a subset of art

## 7. Advanced Configuration Examples

Here are examples of how to configure the RAG system for different providers.

In [16]:
# Configuration examples for different providers

print("🔧 Configuration Examples:")
print("\n1. OpenAI Configuration:")
print("""
# Set environment variable: export OPENAI_API_KEY="your-api-key"
rag_openai = RAGPipeline(
    embedding_provider="openai",
    llm_provider="openai", 
    llm_model="gpt-4"
)
""")

print("\n2. Ollama Configuration (Recommended):")
print("""
# Install Ollama: https://ollama.ai/
# Pull models: 
#   ollama pull llama3.2:1b     # Lightweight, fast model
#   ollama pull nomic-embed-text # Embedding model
rag_ollama = RAGPipeline(
    embedding_provider="ollama",
    llm_provider="ollama",
    llm_model="llama3.2:1b"
)
""")

print("\n3. Mixed Configuration (OpenAI embeddings + Local Ollama LLM):")
print("""
rag_mixed = RAGPipeline(
    embedding_provider="openai",  # High-quality embeddings
    llm_provider="ollama",        # Local inference
    llm_model="llama3.2:1b"       # Fast local model
)
""")

# Utility function to easily switch configurations
def create_rag_system(config_name: str):
    """Create RAG system with predefined configurations."""
    configs = {
        "local": {
            "embedding_provider": "ollama",
            "llm_provider": "ollama",
            "llm_model": "llama3.2:1b"
        },
        "openai": {
            "embedding_provider": "openai", 
            "llm_provider": "openai",
            "llm_model": "gpt-3.5-turbo"
        },
        "mixed": {
            "embedding_provider": "openai",
            "llm_provider": "ollama",
            "llm_model": "llama3.2:1b"
        }
    }
    
    if config_name not in configs:
        raise ValueError(f"Unknown config: {config_name}. Available: {list(configs.keys())}")
    
    config = configs[config_name]
    return RAGPipeline(**config)

print(f"\n✅ Available configurations: {list(['local', 'openai', 'mixed'])}")
print("Usage: create_rag_system('local')")
print("\n💡 llama3.2:1b is perfect for:")
print("  - Fast responses (1B parameters = lightweight)")
print("  - Local development and testing") 
print("  - Low resource usage")
print("  - Quick experimentation with RAG concepts")

🔧 Configuration Examples:

1. OpenAI Configuration:

# Set environment variable: export OPENAI_API_KEY="your-api-key"
rag_openai = RAGPipeline(
    embedding_provider="openai",
    llm_provider="openai", 
    llm_model="gpt-4"
)


2. Ollama Configuration (Recommended):

# Install Ollama: https://ollama.ai/
# Pull models: 
#   ollama pull llama3.2:1b     # Lightweight, fast model
#   ollama pull nomic-embed-text # Embedding model
rag_ollama = RAGPipeline(
    embedding_provider="ollama",
    llm_provider="ollama",
    llm_model="llama3.2:1b"
)


3. Mixed Configuration (OpenAI embeddings + Local Ollama LLM):

rag_mixed = RAGPipeline(
    embedding_provider="openai",  # High-quality embeddings
    llm_provider="ollama",        # Local inference
    llm_model="llama3.2:1b"       # Fast local model
)


✅ Available configurations: ['local', 'openai', 'mixed']
Usage: create_rag_system('local')

💡 llama3.2:1b is perfect for:
  - Fast responses (1B parameters = lightweight)
  - Local developmen

## 8. Summary and Next Steps

### What We've Built

This notebook demonstrates a complete RAG implementation with:

✅ **Flexible Document Chunking**: RecursiveCharacterTextSplitter with configurable chunk sizes  
✅ **Multiple Embedding Options**: Ollama (local) and OpenAI  
✅ **Ephemeral ChromaDB**: In-memory vector store with no background processes  
✅ **Similarity Retrieval**: Vector search with configurable results and scoring  
✅ **Multi-Provider LLM Support**: Ollama (local) and OpenAI  
✅ **Complete RAG Pipeline**: End-to-end question answering with source attribution  

### Key Features

- **No Background Processes**: ChromaDB runs in-memory only
- **Provider Flexibility**: Easy switching between OpenAI and Ollama
- **Modular Design**: Each component can be used independently
- **Rich Retrieval**: Similarity search with scores and metadata
- **Source Attribution**: Retrieved documents linked to answers

### Next Steps

1. **Add your own documents**: Replace sample docs with PDFs, text files, or web content
2. **Experiment with chunking**: Adjust chunk_size and chunk_overlap for your use case  
3. **Try different embeddings**: Compare quality between OpenAI and Ollama
4. **Tune retrieval**: Adjust `k` parameter and similarity thresholds
5. **Customize prompts**: Modify the QA prompt template for better responses
6. **Add persistence**: Use ChromaDB with `persist_directory` for permanent storage

In [ ]:
# Interactive testing cell - try your own questions!

def ask_rag(question: str):
    """Convenient function to query the RAG system."""
    if rag.qa_chain:
        return rag.query(question)
    else:
        print("LLM not available. Showing retrieval results only:")
        docs = rag.retriever.get_relevant_documents(question)
        for i, doc in enumerate(docs, 1):
            source = doc.metadata.get('source', 'unknown')
            print(f"{i}. {source}: {doc.page_content}")
        return docs

# Example usage:
print("🎯 Try asking questions about AI concepts!")
print("Examples:")
print("- ask_rag('What are the benefits of deep learning?')")
print("- ask_rag('How is RAG different from traditional search?')")
print("- ask_rag('What applications use natural language processing?')")
print("\nType your question in the next cell or uncomment one below:")

# Uncomment to test:
# ask_rag("What is the difference between machine learning and deep learning?")